[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/02-data-science-stack/05_sklearn_pipeline_and_leakage.ipynb)

# The Leaky Run and the Honest Run

**Session 7 · companion to HW 1 and HW 2 · nothing here is a homework answer**

A pipeline packages preprocessing and a model so cross-validation can refit
them together inside each training fold. This notebook compares that procedure
with preprocessing fitted before the folds are formed.

Three experiments, in order of how badly each one lies to you:

1. preprocessing fitted before the split — a small, plausible, wrong number,
2. feature selection fitted before the split — a large, exciting, wrong number,
3. a leaky feature — a perfect number that no pipeline can save you from.

Run the cells and watch the scores move.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 500

applicants = pd.DataFrame({
    "income": np.round(rng.lognormal(10.6, 0.45, n)),
    "age": rng.integers(21, 68, n),
    "region": rng.choice(["north", "south", "coast"], n, p=[0.4, 0.35, 0.25]),
    "employment": rng.choice(["salaried", "contract", "self"], n, p=[0.55, 0.25, 0.20]),
})
# Some incomes were never recorded.
applicants.loc[applicants.sample(frac=0.08, random_state=3).index, "income"] = np.nan

logit = (0.9 * (np.log(applicants.income.fillna(applicants.income.median())) - 10.6)
         + 0.35 * (applicants.employment == "salaried")
         - 0.30 * (applicants.employment == "self")
         + rng.normal(0, 0.8, n))
approved = (logit > np.median(logit)).astype(int).to_numpy()   # plain array: positional indexing later

print(applicants.dtypes.to_string())
print("\nrows:", len(applicants), " approved:", approved.mean().round(3),
      " missing incomes:", int(applicants.income.isna().sum()))

income        float64
age             int64
region            str
employment        str

rows: 500  approved: 0.5  missing incomes: 40


The table has two numeric and two categorical columns, with missing incomes.
Our logistic regression needs numeric features without missing values.
ColumnTransformer will prepare each column group inside the training fold.

## The estimator contract, seen once

Before the pipelines, inspect a separate scaler to see fitted attributes such
as `mean_` and `scale_`. This inspection object is never used in evaluation.

In [2]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
print("before fit:", [a for a in vars(scaler) if a.endswith("_")])

scaler.fit(applicants[["age"]])
for name, value in vars(scaler).items():
    if name.endswith("_"):
        print(f"after fit : {name:<18} {np.round(value, 3) if isinstance(value, np.ndarray) and value.dtype.kind == 'f' else value}")

before fit: []
after fit : feature_names_in_  ['age']
after fit : n_features_in_     1
after fit : n_samples_seen_    500.0
after fit : mean_              [43.19]
after fit : var_               [187.334]
after fit : scale_             [13.687]


`mean_` and `scale_` are the fitted state, and they are the whole reason the
test set must never be `fit_transform`ed: doing so replaces the training mean
with a mean computed from data the model is about to be graded on.

## Split first

Everything from here on happens after the split. The test set is set aside now
and opened exactly once, at the bottom of the notebook.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    applicants, approved, test_size=0.25, random_state=0, stratify=approved)
print("train:", X_train.shape, "  test:", X_test.shape)

train: (375, 4)   test: (125, 4)


## Experiment 1: preprocessing fitted on everything

The tempting version: impute and scale the whole table once, then
cross-validate. Every fold's "held-out" rows have already contributed their
values to the imputer's median and the scaler's mean.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

NUMERIC = ["income", "age"]
CATEGORICAL = ["region", "employment"]

prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
])

# WRONG: fit the preprocessing on all of the training data, then cross-validate
# the model on the already-transformed matrix.
leaky_matrix = prep.fit_transform(X_train)
leaky = cross_val_score(LogisticRegression(max_iter=1000), leaky_matrix, y_train, cv=5)

# RIGHT: the preprocessing is a step, so it is refitted inside every fold.
honest_pipe = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))])
honest = cross_val_score(honest_pipe, X_train, y_train, cv=5)

print(f"preprocessing fitted once, then CV : {leaky.mean():.4f}  (+/- {leaky.std():.4f})")
print(f"preprocessing inside the pipeline  : {honest.mean():.4f}  (+/- {honest.std():.4f})")
print(f"difference                         : {leaky.mean() - honest.mean():+.4f}")

preprocessing fitted once, then CV : 0.6880  (+/- 0.0651)
preprocessing inside the pipeline  : 0.6880  (+/- 0.0651)
difference                         : +0.0000


Nothing. Not a small effect — no effect at all, to four decimal places.

Before concluding that one dataset proves anything, measure it properly. The
leak should be worst when the training set is small, because that is when a few
extra rows move a median or a mean the most. Forty random subsamples of sixty
rows each:

In [5]:
advantage = []
for seed in range(40):
    sub = np.random.default_rng(seed).choice(len(X_train), 60, replace=False)
    Xs, ys = X_train.iloc[sub], y_train[sub]
    if len(np.unique(ys)) < 2:
        continue
    leak_score = cross_val_score(LogisticRegression(max_iter=1000),
                                 prep.fit_transform(Xs), ys, cv=4).mean()
    honest_score = cross_val_score(
        Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))]),
        Xs, ys, cv=4).mean()
    advantage.append(leak_score - honest_score)

advantage = np.array(advantage)
print(f"subsamples measured            : {len(advantage)}")
print(f"mean advantage of leaking      : {advantage.mean():+.4f}")
print(f"leaking scored higher in       : {(advantage > 0).mean():.0%} of them")

subsamples measured            : 40
mean advantage of leaking      : -0.0008
leaking scored higher in       : 15% of them


The observed result is that on this data, fitting the imputer and the scaler
before the split buys **nothing** — the difference is noise, and it is not even
reliably in the leak's favour.

Say that out loud, because the usual telling of this story oversells it, and a
student who runs the experiment and sees +0.0000 will conclude the whole warning
was theatre. It is not. The point is that *the size of this leak is a property
of the data, not of your intentions*, and you cannot know which case you are in
until after you have made the mistake. The next experiment is the same mistake
on data that punishes it.

## Experiment 2: selecting features before the split

Add two thousand columns of pure noise and let a filter pick the "best" ones —
using all the labels, before any split.

In [6]:
from sklearn.feature_selection import SelectKBest, f_classif

noise = pd.DataFrame(rng.normal(size=(len(X_train), 2000)),
                     columns=[f"noise_{i}" for i in range(2000)],
                     index=X_train.index)
y_pure = rng.integers(0, 2, len(X_train))     # labels unrelated to ANY column

# WRONG: choose the 20 features that correlate best with y, using all the rows.
chosen = SelectKBest(f_classif, k=20).fit_transform(noise, y_pure)
leaky_selection = cross_val_score(LogisticRegression(max_iter=1000), chosen, y_pure, cv=5)

# RIGHT: selection is a pipeline step, so each fold selects from its own rows.
honest_selection = cross_val_score(
    Pipeline([("select", SelectKBest(f_classif, k=20)),
              ("model", LogisticRegression(max_iter=1000))]),
    noise, y_pure, cv=5)

print("the labels here are random noise; honest accuracy must be about 0.50\n")
print(f"selection before the split : {leaky_selection.mean():.3f}")
print(f"selection inside the fold  : {honest_selection.mean():.3f}")

the labels here are random noise; honest accuracy must be about 0.50

selection before the split : 0.715
selection inside the fold  : 0.472


There is no signal in that data at all — the labels were generated by a coin —
and the leaky procedure still reports an accuracy well above chance. It found
the twenty columns that happen to correlate with these particular labels, and
then measured itself on the same labels.

The selector saw validation labels before CV started. Inside the pipeline,
feature selection uses only each training fold; its score estimates chance
performance, with sampling variation.

## Experiment 3: the leak a pipeline cannot fix

A `Pipeline` controls *when* a computation runs. It knows nothing about what
your columns mean. If a feature is a function of the target — recorded after the
decision, derived from it, or simply the same thing under another name — the
score may be misleadingly high even though the feature is unavailable in production.

In [7]:
with_leak = X_train.copy()
with_leak["days_to_disbursement"] = np.where(y_train == 1,
                                             rng.integers(1, 10, len(X_train)),
                                             -1)     # -1 means: never disbursed

leaky_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]),
         [*NUMERIC, "days_to_disbursement"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])),
    ("model", LogisticRegression(max_iter=1000)),
])

print("cross-validated accuracy WITH the leaky column:",
      round(cross_val_score(leaky_pipe, with_leak, y_train, cv=5).mean(), 4))
print("cross-validated accuracy without it          :",
      round(honest.mean(), 4))

cross-validated accuracy WITH the leaky column: 0.9947
cross-validated accuracy without it          : 0.688


A perfect score, produced by a correctly written pipeline, from a column that
cannot exist at prediction time: the loan has not been disbursed yet when you
are deciding whether to approve it.

The only defence is knowing what each column is and when it was recorded. Ask of
every feature: **would I have this value at the moment I need the prediction?**
If the answer is no, the column is a leak no matter where it sits in your code.

## Spending the test set

Once. Now.

In [8]:
final = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))])
final.fit(X_train, y_train)

print("cross-validated estimate (train only):", round(honest.mean(), 4))
print("held-out test accuracy               :", round(final.score(X_test, y_test), 4))

cross-validated estimate (train only): 0.688
held-out test accuracy               : 0.688


CV and test scores both vary with the sampled rows. A gap in either direction
can reflect sampling variation, distribution differences, or a procedure bug;
the gap alone does not diagnose leakage. Report the final test score after
fixing the modeling choices, together with the training CV estimate and protocol.

**What the fitted pipeline carries with it** — this is why a pipeline is also
the right thing to serialise for Session 29:

In [9]:
import joblib
from pathlib import Path

# Load only your own or otherwise trusted artifacts. Keep library versions.

joblib.dump(final, "approval_pipeline.joblib")

reloaded = joblib.load("approval_pipeline.joblib")
one_row = X_test.iloc[[0]]
print("raw input columns :", list(one_row.columns))
print("prediction        :", reloaded.predict(one_row)[0])
print("the imputer's median travelled with it:",
      round(reloaded.named_steps["prep"].named_transformers_["num"]
            .named_steps["impute"].statistics_[0], 1))
Path("approval_pipeline.joblib").unlink()

raw input columns : ['income', 'age', 'region', 'employment']
prediction        : 1
the imputer's median travelled with it: 41855.0


The reloaded object takes the *raw* table — strings, missing values and all —
and applies the training medians, the training means and the training categories.
That is what prevents training/serving skew, and it is why the artifact you save
is always the pipeline, never the bare model.

## What to take from this

- Fitted state lives on the object with a trailing underscore, and it must come
  from training rows only.
- Any transformation that learns something — an imputer, a scaler, an encoder, a
  feature selector — belongs **inside** the pipeline so it refits per fold.
- Leakage that comes from *ordering* is fixed by a pipeline. Leakage that comes
  from *meaning* is not; only knowing your columns fixes that.
- The test set is opened once, at the end, after every choice has been made.

## Where to go next

- **Reading, Session 7** — the estimator contract, the fit/transform asymmetry,
  and `ColumnTransformer` in full.
- **HW 1 / HW 2** — both assume this pipeline shape. The functions they grade
  are deliberately not written here.

## Reproducing the California housing slide experiment

The earlier experiments use synthetic applicants and classification accuracy.
The deck uses California housing and regression RMSE; their scores are not interchangeable.
This section reproduces the deck's 40 draws with 200 training-pool rows, 150 noise
columns, eight selected features, and 4,000 disjoint evaluation rows per draw.

The dataset downloads once and caches. The filters define a restricted, retrospective
teaching cohort. Extreme ratios are not established errors, and filtering by a known
target cannot define which future unlabeled rows a service will accept.
The five-fold comparisons each train on 160 rows. The independent evaluation uses
models trained on the same 160-row folds, so training-set size is matched.


In [10]:
from sklearn.datasets import fetch_california_housing
from sklearn.feature_selection import f_regression
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold

housing = fetch_california_housing(as_frame=True).frame
clean = housing[(housing.AveRooms < 20) & (housing.AveOccup < 10)
                & (housing.AveBedrms < 3) & (housing.MedHouseVal < 5)].copy()
num_cols = ["MedInc", "HouseAge", "AveRooms", "AveOccup"]
y = clean["MedHouseVal"]
cv5 = KFold(n_splits=5, shuffle=True, random_state=0)
print("restricted cohort:", len(clean))


restricted cohort: 19508


In [11]:
N_SMALL, N_JUNK, K_KEEP, REPS = 200, 150, 8, 40
N_HOLD = 4000
rng = np.random.default_rng(0)

Xc = clean[num_cols + ["Population", "AveBedrms", "Latitude",
                       "Longitude"]].to_numpy()
yc = y.to_numpy()


def leaky_steps():
    return [("scale", StandardScaler()),
            ("select", SelectKBest(f_regression, k=K_KEEP)),
            ("model", Ridge(alpha=1.0))]


RMSE = "neg_root_mean_squared_error"
leaky_runs, honest_runs, truth_runs = [], [], []
for _ in range(REPS):
    idx = rng.choice(len(yc), N_SMALL, replace=False)
    rest = np.setdiff1d(np.arange(len(yc)), idx)
    hold_idx = rng.choice(rest, N_HOLD, replace=False)

    X_small = np.hstack([Xc[idx], rng.normal(size=(N_SMALL, N_JUNK))])
    y_small = yc[idx]
    X_hold = np.hstack([Xc[hold_idx], rng.normal(size=(N_HOLD, N_JUNK))])
    y_hold = yc[hold_idx]

    # WRONG: scale and select on the full sample, then cross-validate.
    X_leaky = StandardScaler().fit_transform(X_small)
    X_leaky = SelectKBest(f_regression,
                          k=K_KEEP).fit_transform(X_leaky, y_small)
    leaky_runs.append(-cross_val_score(Ridge(alpha=1.0), X_leaky, y_small,
                                       cv=cv5, scoring=RMSE).mean())

    # RIGHT: the same steps in a Pipeline, refitted on each training fold.
    honest_runs.append(-cross_val_score(Pipeline(leaky_steps()), X_small,
                                        y_small, cv=cv5,
                                        scoring=RMSE).mean())

    # The arbiter. Cross-validation estimates how a model trained on one
    # fold's training portion generalises, so the fair comparison refits on
    # each of those same portions and scores it on districts that took no
    # part in any of the above.
    fold_rmse = []
    for tr, _ in cv5.split(X_small):
        pred = Pipeline(leaky_steps()).fit(X_small[tr],
                                           y_small[tr]).predict(X_hold)
        fold_rmse.append(root_mean_squared_error(y_hold, pred))
    truth_runs.append(float(np.mean(fold_rmse)))

leaky_rmse = float(np.mean(leaky_runs))
honest_rmse = float(np.mean(honest_runs))
truth_rmse = float(np.mean(truth_runs))

print("\n=== leaky vs honest cross-validation ===")
print(f"{REPS} samples of {N_SMALL} districts, {Xc.shape[1]} real + "
      f"{N_JUNK} uninformative columns, keep k={K_KEEP}")
print(f"leaky  CV RMSE = {leaky_rmse:.3f}  (${leaky_rmse * 100:.1f}k)")
print(f"honest CV RMSE = {honest_rmse:.3f}  (${honest_rmse * 100:.1f}k)")
print(f"held-out RMSE  = {truth_rmse:.3f}  (${truth_rmse * 100:.1f}k), "
      f"{N_HOLD} unseen districts")
print(f"leaky  understates the error by "
      f"${(truth_rmse - leaky_rmse) * 100:.1f}k in RMSE")
print(f"honest is off by "
      f"${abs(truth_rmse - honest_rmse) * 100:.1f}k in RMSE")




=== leaky vs honest cross-validation ===
40 samples of 200 districts, 8 real + 150 uninformative columns, keep k=8
leaky  CV RMSE = 0.694  ($69.4k)
honest CV RMSE = 0.741  ($74.1k)
held-out RMSE  = 0.741  ($74.1k), 4000 unseen districts
leaky  understates the error by $4.8k in RMSE
honest is off by $0.0k in RMSE


The rounded mean RMSEs are 0.694 (leaky CV), 0.741 (honest CV),
and 0.741 (independent evaluation), in $100,000 units. The roughly $4,800 gap
is a difference in aggregate RMSE, not an error reduction promised for every district.
The samples across repetitions can overlap. Close agreement of two estimates
in this seeded experiment does not establish population truth.

Scikit-learn negates RMSE because its scoring interface rewards larger values.
We negate the mean score to recover RMSE; smaller RMSE is better. Feature selection
leaks when validation labels influence which columns survive, even if Ridge itself
never fits those labels. Keep the selector inside each fold.
